## WEIGHT DISTRIBUTION — MASTER CODE (ALL VARIATIONS)

We will:

- Train a small neural network

- Extract weights layer-wise

- Visualize distributions in multiple forms

### Imports & Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.datasets import make_classification

sns.set(style="whitegrid")
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
# Synthetic dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_classes=2,
    random_state=42
)


### Define & Train Neural Network

In [ ]:
model = models.Sequential([
    layers.Input(shape=(20,), name="Input"),
    layers.Dense(64, activation="relu", name="Dense_1"),
    layers.Dense(32, activation="relu", name="Dense_2"),
    layers.Dense(1, activation="sigmoid", name="Output")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

model.fit(X, y, epochs=15, batch_size=32, verbose=0)


### Extract Weights from Each Layer

In [ ]:
weights = {}

for layer in model.layers:
    if hasattr(layer, "kernel"):
        weights[layer.name] = layer.get_weights()[0].flatten()


### Basic Weight Distribution (Histogram)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(
    weights["Dense_1"],
    bins=50,
    alpha=0.7,
    color="steelblue"
)
plt.title("Weight Distribution – Dense_1")
plt.xlabel("Weight value")
plt.ylabel("Frequency")
plt.show()


### KDE (Smooth Density Plot — Recommended)

In [ ]:
plt.figure(figsize=(6, 4))
sns.kdeplot(
    weights["Dense_1"],
    fill=True
)
plt.title("Weight Density – Dense_1")
plt.xlabel("Weight value")
plt.show()


### Weight Distributions Across Layers (Overlay)

In [ ]:
plt.figure(figsize=(7, 5))

for name, w in weights.items():
    sns.kdeplot(w, label=name)

plt.title("Weight Distribution Across Layers")
plt.xlabel("Weight value")
plt.legend()
plt.show()


### Boxplot View (Layer-wise Summary)

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(
    weights.values(),
    labels=weights.keys()
)
plt.title("Weight Distribution (Boxplot)")
plt.ylabel("Weight value")
plt.show()


### Detect Vanishing / Exploding Weights (Quantitative)

In [ ]:
for name, w in weights.items():
    print(
        f"{name:10s} | mean={np.mean(w):+.4f} | std={np.std(w):.4f}"
    )


### Effect of Initialization (He vs Random Normal)

In [ ]:
def build_model(init):
    m = models.Sequential([
        layers.Input(shape=(20,)),
        layers.Dense(64, activation="relu", kernel_initializer=init),
        layers.Dense(32, activation="relu", kernel_initializer=init),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy")
    m.fit(X, y, epochs=3, verbose=0)
    return m

he_model = build_model("he_normal")
rand_model = build_model("random_normal")

he_weights = he_model.layers[1].get_weights()[0].flatten()
rand_weights = rand_model.layers[1].get_weights()[0].flatten()

plt.figure(figsize=(7, 4))
sns.kdeplot(he_weights, label="He Normal")
sns.kdeplot(rand_weights, label="Random Normal")
plt.title("Weight Distribution: He vs Random Init")
plt.xlabel("Weight value")
plt.legend()
plt.show()
